# LimiX Regression Example

This notebook demonstrates how to use the FAIM SDK's **TabularClient** with **LimiX** for tabular regression tasks.

[LimiX](https://github.com/limix-ldm/LimiX) is a foundation model for tabular machine learning that supports both classification and regression.

## Setup

Install dependencies and import required libraries.

In [70]:
import os

import numpy as np
from sklearn.datasets import fetch_california_housing

from faim_sdk import LimiXPredictRequest, TabularClient

## Load and Prepare Data

Load the California housing regression dataset from scikit-learn.

In [71]:
# Load California housing dataset
house_data = fetch_california_housing()
X, y = house_data.data, house_data.target

# Create a simple 80/20 split for demonstration
split_idx = int(0.8 * len(X))
X_train = X[:split_idx].astype(np.float32)
y_train = y[:split_idx].astype(np.float32)
X_test = X[split_idx:].astype(np.float32)
y_test = y[split_idx:].astype(np.float32)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Target range: [{y_train.min():.2f}, {y_train.max():.2f}]")

Training set size: (16512, 8)
Test set size: (4128, 8)
Number of features: 8
Target range: [0.15, 5.00]


## Initialize TabularClient

Create a client to interact with the LimiX model.

In [72]:
# Initialize the client
client = TabularClient(
    base_url="https://api.faim.it.com",
    api_key=os.environ.get("FAIM_API_KEY"),  # Replace with your actual API key
    timeout=300.0,
)

print("TabularClient initialized!")

TabularClient initialized!


## Create Regression Request

Prepare a LimiX regression request.

In [73]:
# Create a LimiX regression request
request = LimiXPredictRequest(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    task_type="Regression",
    use_retrieval=False,
)

print("Request prepared:")
print(f"  X_train shape: {request.X_train.shape}")
print(f"  X_test shape: {request.X_test.shape}")
print(f"  Task type: {request.task_type}")

Request prepared:
  X_train shape: (16512, 8)
  X_test shape: (4128, 8)
  Task type: Regression


## Make Predictions

Send the request to LimiX and get regression predictions.

In [74]:
try:
    # Make predictions
    response = client.predict(request)

    print(f"Predictions shape: {response.predictions.shape}")
    print(f"First 10 predictions: {response.predictions[:10]}")
    print(f"Prediction range: [{response.predictions.min():.2f}, {response.predictions.max():.2f}]")

except Exception as e:
    print(f"Error: {e}")
    print("\nMake sure your API key is valid and the service is available.")

Predictions shape: (4128,)
First 10 predictions: [1.25906553 1.422029   1.09111338 1.11938256 1.27015148 1.11827396
 1.13822867 1.0872333  1.09831925 1.20030999]
Prediction range: [0.51, 5.06]


## Evaluate Results

Calculate regression metrics.

In [75]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    y_pred = response.predictions

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("Regression Metrics:")
    print(f"  MSE:  {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R²:   {r2:.4f}")
except NameError:
    print("Run prediction cell first to evaluate metrics.")

Regression Metrics:
  MSE:  0.5715
  RMSE: 0.7560
  MAE:  0.5277
  R²:   0.6078


## Residual Analysis

Analyze prediction errors.

In [76]:
try:
    residuals = y_test - y_pred

    print("Residual Statistics:")
    print(f"  Mean:   {residuals.mean():.4f}")
    print(f"  Std:    {residuals.std():.4f}")
    print(f"  Min:    {residuals.min():.4f}")
    print(f"  Max:    {residuals.max():.4f}")
except NameError:
    print("Run prediction cell first.")

Residual Statistics:
  Mean:   0.2896
  Std:    0.6983
  Min:    -2.8715
  Max:    3.9316
